## Airline AI Assistant

In [1]:
import os
import json
from openai import OpenAI
import gradio as gr

In [2]:
MODELllama = 'llama3.2:latest'
MODELPhi = 'phi3:latest'
Model_gpt_oss = 'gpt-oss:20b'
Model_gemma = 'gemma3:270m'
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

#if not running, run "ollama serve" at a command line

In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chat_with_history(message, history):
    history = [{"role" : h["role"], "content": h["content"]} for h in history]
    message = [{"role" : "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model= MODELllama, messages= message, stream= True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response 

In [5]:
gr.ChatInterface(fn = chat_with_history).launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

In [6]:
ticket_prices = {"london": "$799", "paris": "$199", "tokyo": "$399", "dubai": "$99"}

In [7]:
def get_ticket_price(destination_city): 
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

In [8]:
get_ticket_price("tokyo")

Tool called for city tokyo


'The price of a ticket to tokyo is $399'

In [9]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [10]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [11]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [12]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=Model_gpt_oss, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        tool_msg = handle_tool_call(assistant_msg)
        messages.append(assistant_msg)
        messages.append(tool_msg)

        # for message in messages:
        #     print(message)
        
        response = openai.chat.completions.create(model=Model_gpt_oss, messages=messages)

    return response.choices[0].message.content

In [13]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


## couple of improvements:
- Handling multiple tool_calls in 1 response
- Handling multiple tool_calls one after another

In [14]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [15]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=Model_gpt_oss, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=Model_gpt_oss, messages=messages)
    
    return response.choices[0].message.content

In [16]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


the above will work only for one city destination : what is the ticket price for london?

to support multiple city dest, use "for loop": 

In [17]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

ask : what is the ticket price for london and dubai?

but this wont support question like: 
please check the ticket price for london only if its under $1000 then please check for paris. 

to support this use "while loop"

In [18]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=Model_gpt_oss, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=Model_gpt_oss, messages=messages, tools=tools)
    
    return response.choices[0].message.content

## Database - instead of python dictionary

In [19]:
import sqlite3

In [20]:
# create a local DB

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [23]:
def get_ticket_price(dest_city):
    print(f"DATABASE TOOL CALLED: Getting price for {dest_city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (dest_city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {dest_city} is ${result[0]}" if result else "No price data available for this city"

In [24]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [25]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices(city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [27]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [28]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'Ticket price to London is $799.0'

In [30]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for London
